# Module 1: Spark Basics Refresher

**Objective**: Get comfortable with SparkSession, DataFrames, and basic operations.

**Time**: ~1-2 hours

---

## What You'll Learn
1. Creating and configuring a SparkSession
2. Reading data from CSV files
3. Basic DataFrame operations (select, filter, show)
4. Schema inference vs explicit schema
5. Writing data to Parquet format

# Remote SparkSession via Databricks Connect
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.serverless().getOrCreate()
print(f"✅ Spark {spark.version} on Databricks serverless")

### 💡 Key Concepts

| Component | Description |
|-----------|-------------|
| `appName` | Name shown in Spark UI |
| `master("local[*]")` | Use all CPU cores locally |
| `spark.sql.shuffle.partitions` | Partitions after shuffle (default 200 is too high for local) |
| `spark.driver.memory` | Memory for the driver JVM |

## 2. Reading Data

Let's read the synthetic banking data we generated.

In [1]:
import sys; sys.path.insert(0, '..')
from notebooks.notebook_setup import spark, customers_df, accounts_df, transactions_df, branches_df, MODE

✅ Databricks Connect | Spark 4.1.0
   Tracking: https://dbc-cdbdfd07-5797.cloud.databricks.com → Job runs
📦 Reading Parquet from S3...
✅ Data loaded:
   Branches:          100
   Customers:      10,000
   Accounts:       16,442
   Transactions: 5,000,000


In [15]:
from pyspark.sql.functions import to_date, col
S3_RAW   = "s3a://sparkling-data-test/data/raw"
transactions_df \
    .withColumn("txn_date", to_date(col("txn_datetime"))) \
    .write \
    .mode("overwrite") \
    .partitionBy("txn_date") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .save(f"{S3_RAW}/transactions_delta")

In [12]:
transactions_df.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_datetime: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- reference: string (nullable = true)
 |-- description: string (nullable = true)



In [ ]:
from delta.tables import DeltaTable
S3_RAW   = "s3a://sparkling-data-test/data/raw"

dt = DeltaTable.forPath(spark, f"{S3_RAW}/transactions_delta")
dt.detail().select("numFiles", "sizeInBytes").show()

# See the per-file stats Delta maintains
spark.sql(f"DESCRIBE DETAIL delta.`{S3_RAW}/transactions_delta`").show(truncate=False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|     365|  165776765|
+--------+-----------+

+------+------------------------------------+----+-----------+-----------------------------------------------------+-----------------------+-------------------+----------------+-----------------+--------+-----------+-------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name|description|location                                             |createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                           |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                                     |clusterByAuto|
+------+------------------------------------+----+-----------+----

In [16]:
transactions_delta_df = spark.read.format("delta").load(f"{S3_RAW}/transactions_delta")
transactions_delta_df.show(10)

+----------+----------+-------------------+-----------+--------------+--------+----------------+-----------------+---------+------------+--------------------+----------+
|    txn_id|account_id|       txn_datetime|   txn_type|        amount|currency|         channel|merchant_category|   status|   reference|         description|  txn_date|
+----------+----------+-------------------+-----------+--------------+--------+----------------+-----------------+---------+------------+--------------------+----------+
|TXN0002838|ACCT000740|2025-11-04 21:19:19|        Fee|      12592.31|     VND|Internet Banking|             NULL|Completed|REF931434515|     Fee transaction|2025-11-04|
|TXN0006758|ACCT009332|2025-11-04 20:34:28|        Fee|      61861.42|     VND|             POS|             NULL|Completed|REF461026093|     Fee transaction|2025-11-04|
|TXN0007974|ACCT014643|2025-11-04 01:26:45| Withdrawal| 3.703897104E7|     VND|             ATM|             NULL|Completed|REF137355284|Withdrawal tr

In [17]:
# Compact small files into larger ones
spark.sql("""
    OPTIMIZE delta.`s3a://sparkling-data-test/data/raw/transactions_delta`
    ZORDER BY (account_id)
""")

# Remove old file versions (default keeps 30 days)
spark.sql("""
    VACUUM delta.`s3a://sparkling-data-test/data/raw/transactions_delta`
    RETAIN 168 HOURS
""")

DataFrame[path: string]

In [3]:
# Check the inferred schema
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- nationality: string (nullable = true)



In [4]:
import pyspark.sql.functions as F
# Basic statistics
print(f"Total customers: {customers_df.count():,}")
print(f"Columns: {len(customers_df.columns)}")
print(f"Partitions: {customers_df.select(F.spark_partition_id()).distinct().count()}")

Total customers: 10,000
Columns: 10
Partitions: 8


In [5]:
# Basic statistic for transactions

print(f"Total transactions: {transactions_df.count():,}")
print(f"Columns: {len(transactions_df.columns)}")
print(f"Partitions: {transactions_df.select(F.spark_partition_id()).distinct().count()}")


Total transactions: 5,000,000
Columns: 11
Partitions: 8


In [6]:
transactions_df.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_datetime: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- reference: string (nullable = true)
 |-- description: string (nullable = true)



In [7]:
transactions_df.groupBy("txn_type").count().explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- == Initial Plan ==
   PhotonResultStage (8)
   +- PhotonColumnarToRow (7)
      +- PhotonGroupingAgg (6)
         +- PhotonShuffleExchangeSource (5)
            +- PhotonShuffleMapStage (4)
               +- PhotonShuffleExchangeSink (3)
                  +- PhotonGroupingAgg (2)
                     +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [1]: [txn_type#13211]
Location: InMemoryFileIndex [s3a://sparkling-data-test/data/raw/transactions]
ReadSchema: struct<txn_type:string>

(2) PhotonGroupingAgg
Input [1]: [txn_type#13211]
Arguments: [txn_type#13211], [partial_count(1) AS count#13255L], [count#13254L], [txn_type#13211, count#13255L], false

(3) PhotonShuffleExchangeSink
Input [2]: [txn_type#13211, count#13255L]
Arguments: hashpartitioning(txn_type#13211, 159)

(4) PhotonShuffleMapStage
Input [2]: [txn_type#13211, count#13255L]
Arguments: ENSURE_REQUIREMENTS, [id=#9093]

(5) PhotonShuffleExchangeSource
Input [2]: [txn_type

In [8]:
import time
start = time.time()
count = transactions_df.count()
elapsed = time.time() - start
print(f"Transactions: {count:,} | Time: {elapsed:.1f}s")

Transactions: 5,000,000 | Time: 0.5s


## 3. Explicit Schema Definition

Schema inference is convenient but slow for large files. Explicit schemas are faster and more reliable.

In [9]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType
)

# Define explicit schema for accounts
accounts_schema = StructType([
    StructField("account_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("branch_id", StringType(), False),
    StructField("account_type", StringType(), True),
    StructField("balance", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("opened_date", StringType(), True),
    StructField("last_activity_date", StringType(), True)
])


accounts_df.printSchema()
accounts_df.show(5)

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- opened_date: string (nullable = true)
 |-- last_activity_date: string (nullable = true)

+----------+-----------+---------+------------+-------------+--------+------+-----------+------------------+
|account_id|customer_id|branch_id|account_type|      balance|currency|status|opened_date|last_activity_date|
+----------+-----------+---------+------------+-------------+--------+------+-----------+------------------+
|ACCT012332| CUST007475| BR000003|  Investment|   8398262.39|     VND|Active| 2023-11-11|        2025-11-17|
|ACCT012333| CUST007475| BR000037|     Savings|1.212603567E7|     VND|Closed| 2019-07-02|        2025-09-09|
|ACCT012334| CUST007476| BR000041|  Investment|1.889268539E7|     VND

## 4. Basic DataFrame Operations

### 4.1 Selecting Columns

In [10]:
from pyspark.sql.functions import col

# Method 1: String column names
customers_df.select("customer_id", "name", "segment").show(5)

# Method 2: Using col() function
customers_df.select(col("customer_id"), col("name"), col("segment")).show(5)

# Method 3: Using DataFrame reference
customers_df.select(customers_df.customer_id, customers_df.name).show(5)

+-----------+--------------+-------------+
|customer_id|          name|      segment|
+-----------+--------------+-------------+
| CUST006251|Customer 06251|         Mass|
| CUST006252|Customer 06252|Mass Affluent|
| CUST006253|Customer 06253|         Mass|
| CUST006254|Customer 06254|Mass Affluent|
| CUST006255|Customer 06255|          HNW|
+-----------+--------------+-------------+
only showing top 5 rows
+-----------+--------------+-------------+
|customer_id|          name|      segment|
+-----------+--------------+-------------+
| CUST006251|Customer 06251|         Mass|
| CUST006252|Customer 06252|Mass Affluent|
| CUST006253|Customer 06253|         Mass|
| CUST006254|Customer 06254|Mass Affluent|
| CUST006255|Customer 06255|          HNW|
+-----------+--------------+-------------+
only showing top 5 rows
+-----------+--------------+
|customer_id|          name|
+-----------+--------------+
| CUST006251|Customer 06251|
| CUST006252|Customer 06252|
| CUST006253|Customer 06253|
| CU

### 4.2 Filtering Rows

In [11]:
# Filter high-net-worth customers
hnw_customers = customers_df.filter(col("segment").isin(["HNW", "UHNW"]))
print(f"HNW/UHNW Customers: {hnw_customers.count():,}")
hnw_customers.show(5)

HNW/UHNW Customers: 1,015
+-----------+--------------+--------------------+----------+-------+-----------------+----------+-------------+------+-----------+
|customer_id|          name|               email|     phone|segment|registration_date|kyc_status|date_of_birth|gender|nationality|
+-----------+--------------+--------------------+----------+-------+-----------------+----------+-------------+------+-----------+
| CUST006255|Customer 06255|customer6255@exam...|0965652271|    HNW|       2019-02-06|  Verified|   1986-05-15|     M| Vietnamese|
| CUST006271|Customer 06271|customer6271@exam...|0996171947|   UHNW|       2019-12-19|  Verified|   1964-06-27|     F| Vietnamese|
| CUST006281|Customer 06281|customer6281@exam...|0932432040|    HNW|       2022-03-24|  Verified|   1985-05-22|     F| Vietnamese|
| CUST006282|Customer 06282|customer6282@exam...|0937205823|    HNW|       2021-08-14|  Verified|   1990-10-14|     F| Vietnamese|
| CUST006288|Customer 06288|customer6288@exam...|09301640

In [12]:
# Multiple conditions
verified_hnw = customers_df.filter(
    (col("segment") == "HNW") & 
    (col("kyc_status") == "Verified")
)
print(f"Verified HNW: {verified_hnw.count():,}")

Verified HNW: 597


### 4.3 Sorting

In [13]:
# Sort accounts by balance (descending)
accounts_df.orderBy(col("balance").desc()).show(10)

accounts_df.orderBy(col("balance").desc()).explain(mode="formatted")

+----------+-----------+---------+------------+-----------------+--------+------+-----------+------------------+
|account_id|customer_id|branch_id|account_type|          balance|currency|status|opened_date|last_activity_date|
+----------+-----------+---------+------------+-----------------+--------+------+-----------+------------------+
|ACCT011954| CUST007252| BR000005|  Investment|4.995375548614E10|     VND|Active| 2018-09-11|        2025-11-16|
|ACCT010437| CUST006307| BR000061|  Investment|4.990152958166E10|     VND|Active| 2018-06-02|        2025-11-06|
|ACCT012384| CUST007505| BR000030|     Savings|4.987006796991E10|     VND|Active| 2018-01-20|        2025-09-24|
|ACCT005375| CUST003265| BR000008|      Credit|4.982045046904E10|     VND|Active| 2023-08-08|        2024-11-27|
|ACCT013500| CUST008207| BR000068|    Checking|4.971507838614E10|     VND|Active| 2023-09-23|        2024-10-27|
|ACCT001016| CUST000621| BR000024|      Credit| 4.97019842644E10|     VND|Closed| 2019-06-06|   

### 4.4 Distinct Values

In [14]:
# Get unique segments
customers_df.select("segment").distinct().show()

# Count by segment
customers_df.groupBy("segment").count().orderBy("count", ascending=False).show()

+-------------+
|      segment|
+-------------+
|         Mass|
|Mass Affluent|
|          HNW|
|     Affluent|
|         UHNW|
+-------------+

+-------------+-----+
|      segment|count|
+-------------+-----+
|         Mass| 4999|
|Mass Affluent| 2445|
|     Affluent| 1541|
|          HNW|  703|
|         UHNW|  312|
+-------------+-----+



## 5. Lazy Evaluation Demo

**Key Concept**: Transformations (filter, select, join) are LAZY - they don't execute until an ACTION is called.

In [15]:
# These are all TRANSFORMATIONS (lazy, no execution yet)
result = customers_df \
    .filter(col("segment") == "Affluent") \
    .select("customer_id", "name", "segment") \
    .orderBy("name")

print("Transformations defined (nothing executed yet)")
print(f"Query plan ready: {result.isLocal}")

Transformations defined (nothing executed yet)
Query plan ready: <bound method DataFrame.isLocal of DataFrame[customer_id: string, name: string, segment: string]>


In [16]:
# View the execution plan
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [name#13180 ASC NULLS FIRST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#9759]
               +- PhotonShuffleExchangeSink rangepartitioning(name#13180 ASC NULLS FIRST, 16)
                  +- PhotonScan parquet [customer_id#13179,name#13180,segment#13183] DataFilters: [isnotnull(segment#13183), (segment#13183 = Affluent)], DictionaryFilters: [(segment#13183 = Affluent)], Format: parquet, Location: InMemoryFileIndex(1 paths)[s3a://sparkling-data-test/data/raw/customers], OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<customer_id:string,name:string,segment:string>, RequiredDataFilters: [isnotnull(segment#13183), (segment#13183 = Affluent)]


== Photon Explanation ==
The query is fully supported by Photon.


In [17]:
# NOW it executes (show() is an ACTION)
result.show(5)

+-----------+--------------+--------+
|customer_id|          name| segment|
+-----------+--------------+--------+
| CUST000003|Customer 00003|Affluent|
| CUST000004|Customer 00004|Affluent|
| CUST000005|Customer 00005|Affluent|
| CUST000012|Customer 00012|Affluent|
| CUST000016|Customer 00016|Affluent|
+-----------+--------------+--------+
only showing top 5 rows


### Actions vs Transformations

| Transformations (Lazy) | Actions (Trigger Execution) |
|------------------------|-----------------------------|
| `select()` | `show()` |
| `filter()` | `count()` |
| `groupBy()` | `collect()` |
| `join()` | `take(n)` |
| `orderBy()` | `write.*` |

## 6. Writing Data to Parquet

Parquet is a columnar format - faster and smaller than CSV.

In [18]:
# Replace the output path with S3
OUTPUT_PATH = "s3a://sparkling-data-test/data/processed"
# Write customers to Parquet
customers_df.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/customers_parquet")

print("✅ Customers saved to Parquet!")

✅ Customers saved to Parquet!


In [20]:
# Read back and verify
customers_parquet = spark.read.parquet(f"{OUTPUT_PATH}/customers_parquet")
print(f"Rows: {customers_parquet.count():,}")
customers_parquet.show(3)

Rows: 10,000
+-----------+--------------+--------------------+----------+-------+-----------------+----------+-------------+------+-----------+
|customer_id|          name|               email|     phone|segment|registration_date|kyc_status|date_of_birth|gender|nationality|
+-----------+--------------+--------------------+----------+-------+-----------------+----------+-------------+------+-----------+
| CUST007501|Customer 07501|customer7501@exam...|0976501943|   Mass|       2017-04-18|  Verified|   1989-02-05|     M| Vietnamese|
| CUST007502|Customer 07502|customer7502@exam...|0968501018|   Mass|       2017-10-13|  Verified|   1984-09-21|     M| Vietnamese|
| CUST007503|Customer 07503|customer7503@exam...|0945582644|   Mass|       2017-09-19|   Pending|   1976-03-13|     F| Vietnamese|
+-----------+--------------+--------------------+----------+-------+-----------------+----------+-------------+------+-----------+
only showing top 3 rows


## 7. Spark SQL

You can also use SQL syntax directly!

In [21]:
# Register DataFrame as a temp view
customers_df.createOrReplaceTempView("customers")
accounts_df.createOrReplaceTempView("accounts")

# Now use SQL!
spark.sql("""
    SELECT segment, COUNT(*) as customer_count
    FROM customers
    GROUP BY segment
    ORDER BY customer_count DESC
""").show()

+-------------+--------------+
|      segment|customer_count|
+-------------+--------------+
|         Mass|          4999|
|Mass Affluent|          2445|
|     Affluent|          1541|
|          HNW|           703|
|         UHNW|           312|
+-------------+--------------+



In [22]:
# Join customers and accounts using SQL
spark.sql("""
    SELECT 
        c.name,
        c.segment,
        a.account_type,
        a.balance
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    ORDER BY a.balance DESC
    LIMIT 10
""").show()

+--------------+-------+------------+-----------------+
|          name|segment|account_type|          balance|
+--------------+-------+------------+-----------------+
|Customer 07252|   UHNW|  Investment|4.995375548614E10|
|Customer 06307|   UHNW|  Investment|4.990152958166E10|
|Customer 07505|   UHNW|     Savings|4.987006796991E10|
|Customer 03265|   UHNW|      Credit|4.982045046904E10|
|Customer 08207|   UHNW|    Checking|4.971507838614E10|
|Customer 00621|   UHNW|      Credit| 4.97019842644E10|
|Customer 01036|   UHNW|      Credit|4.967998201058E10|
|Customer 05095|   UHNW|  Investment|4.927236387187E10|
|Customer 01242|   UHNW|Term Deposit|4.925558412985E10|
|Customer 06089|   UHNW|  Investment| 4.92261121045E10|
+--------------+-------+------------+-----------------+



## ✅ Practice Exercises

Complete these exercises to solidify your understanding:

In [24]:
# Exercise 1: Read transactions.csv and show the schema
# Your code here:
transactions_df.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_datetime: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- reference: string (nullable = true)
 |-- description: string (nullable = true)



import os

# Get file size in bytes
# file_path = str(DATA_PATH / "transactions.csv")
# file_size_bytes = os.path.getsize(file_path)

# # Get Spark DataFrame estimated size in bytes
# # This uses the logical plan statistics
df_size_bytes = transactions_df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()

print(f"File size (CSV): {file_size_bytes / (1024**2):.2f} MB")
print(f"DataFrame size (Memory): {df_size_bytes / (1024**2):.2f} MB")
print(f"Comparison ratio: {df_size_bytes / file_size_bytes:.2f}x")


In [ ]:
transactions_delta_df.limit(1000).show()

transactions_df.limit(1000).show()
# .collect() is slow because it is an action that:
# 1. Triggers the execution of all preceding transformations in the Spark DAG.
# 2. Transfers all data from distributed executors to the single driver node over the network.
# 3. Requires the driver to have enough memory to store the entire dataset, which can lead to OOM errors.
# 4. Involves significant serialization and deserialization overhead.

# .take(1000) is faster because it:
# 1. Only transfers a small subset of data (1000 rows) to the driver node.
# 2. Avoids the need for full dataset transfer and memory constraints.
# 3. Is less resource-intensive in terms of network and memory usage.
# 4. Is still a transformation (lazy), so it doesn't trigger execution until an action is called.


[Row(txn_id='TXN0002838', account_id='ACCT000740', txn_datetime='2025-11-04 21:19:19', txn_type='Fee', amount=12592.31, currency='VND', channel='Internet Banking', merchant_category=None, status='Completed', reference='REF931434515', description='Fee transaction', txn_date=datetime.date(2025, 11, 4)),
 Row(txn_id='TXN0006758', account_id='ACCT009332', txn_datetime='2025-11-04 20:34:28', txn_type='Fee', amount=61861.42, currency='VND', channel='POS', merchant_category=None, status='Completed', reference='REF461026093', description='Fee transaction', txn_date=datetime.date(2025, 11, 4)),
 Row(txn_id='TXN0007974', account_id='ACCT014643', txn_datetime='2025-11-04 01:26:45', txn_type='Withdrawal', amount=37038971.04, currency='VND', channel='ATM', merchant_category=None, status='Completed', reference='REF137355284', description='Withdrawal transaction', txn_date=datetime.date(2025, 11, 4)),
 Row(txn_id='TXN0008438', account_id='ACCT000316', txn_datetime='2025-11-04 12:15:42', txn_type='Fee

In [22]:
import time

S3_RAW = "s3a://sparkling-data-test/data/raw"

# ── Read both tables ──
parquet_df = spark.read.parquet(f"{S3_RAW}/transactions")
delta_df   = spark.read.format("delta").load(f"{S3_RAW}/transactions_delta")

# ── Benchmark: Parquet (non-Delta) ──
print("=" * 60)
print("📊 PARQUET (non-Delta)")
print("=" * 60)

start = time.time()
parquet_df.limit(1000).show()
parquet_time = time.time() - start
print(f"⏱️  Execution time: {parquet_time:.2f}s\n")

print("Execution Plan:")
parquet_df.limit(1000).explain(True)

# ── Benchmark: Delta ──
print("\n" + "=" * 60)
print("📊 DELTA")
print("=" * 60)

start = time.time()
delta_df.limit(1000).show()
delta_time = time.time() - start
print(f"⏱️  Execution time: {delta_time:.2f}s\n")

print("Execution Plan:")
delta_df.limit(1000).explain(True)

# ── Summary ──
print("\n" + "=" * 60)
print("📋 COMPARISON SUMMARY")
print("=" * 60)
print(f"  Parquet: {parquet_time:.2f}s")
print(f"  Delta:   {delta_time:.2f}s")
faster = "Delta" if delta_time < parquet_time else "Parquet"
speedup = max(parquet_time, delta_time) / min(parquet_time, delta_time)
print(f"  Winner:  {faster} ({speedup:.1f}x faster)")

📊 PARQUET (non-Delta)
+----------+----------+-------------------+------------+--------------+--------+----------------+-----------------+---------+------------+--------------------+
|    txn_id|account_id|       txn_datetime|    txn_type|        amount|currency|         channel|merchant_category|   status|   reference|         description|
+----------+----------+-------------------+------------+--------------+--------+----------------+-----------------+---------+------------+--------------------+
|TXN0000006|ACCT002416|2025-11-23 22:29:27|Transfer Out| 5.715450649E7|     VND|          Branch|             NULL|Completed|REF531822652|Transfer Out tran...|
|TXN0000022|ACCT002074|2025-03-08 23:49:01|         Fee|      32578.82|     VND|             ATM|             NULL|Completed|REF860012904|     Fee transaction|
|TXN0000038|ACCT001461|2025-12-22 00:22:18|    Interest|1.7680802708E8|     VND|             API|             NULL|Completed|REF949828865|Interest transaction|
|TXN0000054|ACCT01

In [27]:
transactions_df.take(5)

[Row(txn_id='TXN0000006', account_id='ACCT002416', txn_datetime='2025-11-23 22:29:27', txn_type='Transfer Out', amount=57154506.49, currency='VND', channel='Branch', merchant_category=None, status='Completed', reference='REF531822652', description='Transfer Out transaction'),
 Row(txn_id='TXN0000022', account_id='ACCT002074', txn_datetime='2025-03-08 23:49:01', txn_type='Fee', amount=32578.82, currency='VND', channel='ATM', merchant_category=None, status='Completed', reference='REF860012904', description='Fee transaction'),
 Row(txn_id='TXN0000038', account_id='ACCT001461', txn_datetime='2025-12-22 00:22:18', txn_type='Interest', amount=176808027.08, currency='VND', channel='API', merchant_category=None, status='Completed', reference='REF949828865', description='Interest transaction'),
 Row(txn_id='TXN0000054', account_id='ACCT014352', txn_datetime='2025-02-22 17:53:58', txn_type='Deposit', amount=49477702.2, currency='VND', channel='ATM', merchant_category=None, status='Completed', re

In [23]:
# Exercise 2: Count total transactions
# Your code here:
CountTotalTransactions = transactions_df.count()
print(f"Total number of transactions: {CountTotalTransactions}")

Total number of transactions: 5000000


In [4]:
# Exercise 3: Find all "Failed" transactions
# Your code here:
FailedTransactions = transactions_df.filter("status='Failed'")
FailedTransactions.show()


NameError: name 'transactions_df' is not defined

In [25]:
# Exercise 4: Group transactions by channel and count
# Your code here:
TransactionsGroupedByChannel = transactions_df.groupBy("channel").count()
TransactionsGroupedByChannel.show()

+----------------+------+
|         channel| count|
+----------------+------+
|      Mobile App|832776|
|             POS|832721|
|          Branch|834369|
|Internet Banking|833933|
|             API|832662|
|             ATM|833539|
+----------------+------+



In [26]:
# Exercise 5: Write transactions to Parquet, partitioned by txn_type
# Hint: Use .partitionBy("txn_type")
# Your code here:
OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Write customers to Parquet
transactions_df.repartition(100).write \
    .mode("overwrite") \
    .partitionBy("txn_type") \
    .parquet(str(OUTPUT_PATH / "transactions_parquet"))

print("✅ Customers saved to Parquet!")

✅ Customers saved to Parquet!


## 🎯 Key Takeaways

1. **SparkSession** is your entry point - configure memory and partitions for local dev
2. **Explicit schemas** are faster than inferSchema for large files
3. **Lazy evaluation** means transformations only execute when an action is called
4. **Parquet** is better than CSV for Spark workloads (columnar, compressed)
5. **Spark SQL** lets you use familiar SQL syntax on DataFrames

---

**Next**: [[02_banking_transformations.ipynb]] - Real banking transformations

In [27]:
# Clean up
spark.stop()
print("✅ Spark session stopped.")

✅ Spark session stopped.
